<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 6 · Lunes — Pipelines y ColumnTransformer</h1>
<h3>Automatizando el flujo de Machine Learning</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final vas a poder:

1. Entender **qué problema resuelven los Pipelines** y por qué son esenciales.
2. Usar **`Pipeline`** y **`make_pipeline`** para encadenar pasos.
3. Usar **`ColumnTransformer`** para aplicar transformaciones distintas a columnas numéricas y categóricas.
4. Combinar todo en un flujo profesional: encoding + estandarización + modelo en una sola línea de `fit`.
5. Resolver **3 ejercicios** integradores.

# 1. ¿Por qué necesitamos Pipelines?

Hasta ahora, cuando entrenamos un modelo hacemos esto:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, ...)

scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc  = scaler.transform(X_test)

modelo = LinearRegression()
modelo.fit(X_train_esc, y_train)
y_pred = modelo.predict(X_test_esc)
```

### Problemas reales con este flujo:

1. 🐛 **Es fácil olvidar pasos** — ¿estandarizaste el test? ¿usaste `fit_transform` o `transform`?
2. 🚨 **Data leakage accidental** — si haces `scaler.fit(X)` antes del split, contaminas el modelo.
3. 🔄 **Difícil de reproducir** — al subir a producción tienes que recordar todo el orden.
4. 🧪 **Imposible de optimizar fácilmente** — no puedes hacer GridSearch sobre el scaler + modelo a la vez.

### 🧠 Analogía simple

Un **Pipeline** es como una **línea de ensamblaje** en una fábrica de autos: cada paso (pintar, ensamblar, probar) ocurre en orden, sin que el operario tenga que recordar manualmente cada paso. Solo dices *"empezar"* y todo fluye.

En sklearn:

```python
pipeline = make_pipeline(StandardScaler(), LinearRegression())
pipeline.fit(X_train, y_train)   # ← UNA SOLA línea
y_pred = pipeline.predict(X_test) # ← UNA SOLA línea
```

Más limpio, más seguro, más profesional.

# 2. `make_pipeline` y `Pipeline` — encadenar pasos

Sklearn ofrece **dos formas** de crear pipelines:

| Función | Sintaxis | Cuándo usar |
|---|---|---|
| `make_pipeline(step1, step2, ...)` | Sin nombres — los asigna automáticamente | Para flujos simples y rápidos |
| `Pipeline([('nombre1', step1), ('nombre2', step2)])` | Con nombres explícitos | Cuando quieres usar GridSearch sobre los pasos |

Ambos hacen lo mismo. Vamos con `make_pipeline` (más simple).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn import set_config

# Para visualizar pipelines como diagrama
set_config(display='diagram')

# Cargamos mpg (auto-mpg)
mpg = sns.load_dataset('mpg').dropna()
X = mpg.drop(columns=['mpg', 'origin', 'name'])  # solo numéricas por simplicidad
y = mpg['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline simple con make_pipeline
pipe = make_pipeline(StandardScaler(), LinearRegression())
pipe

In [ ]:
# UNA SOLA línea para fit + transform + entrenar
pipe.fit(X_train, y_train)

# Y otra sola línea para predict (transforma + predice automáticamente)
y_pred = pipe.predict(X_test)

print(f'R²: {pipe.score(X_test, y_test):.4f}')

In [ ]:
# Mismo pipeline pero con nombres explícitos (útil para GridSearch)
pipe_named = Pipeline([
    ('escalador', StandardScaler()),
    ('modelo',    LinearRegression())
])

pipe_named.fit(X_train, y_train)
print(f'R²: {pipe_named.score(X_test, y_test):.4f}')

# Podemos acceder a cualquier paso por nombre
print(f'\nMedia aprendida por el escalador: {pipe_named.named_steps["escalador"].mean_[:3]}...')
print(f'Coeficientes del modelo: {pipe_named.named_steps["modelo"].coef_[:3]}...')

# 3. `ColumnTransformer` — transformaciones distintas por columna

Un Pipeline aplica los mismos pasos a TODAS las columnas. Pero en la vida real tienes columnas mixtas:

- **Numéricas** (`age`, `salary`) → quieres **estandarizar** (`StandardScaler`)
- **Categóricas** (`city`, `gender`) → quieres **codificar** (`OneHotEncoder`)

Aplicar `StandardScaler` a `'Santiago'` o `OneHotEncoder` a `25` no tiene sentido. Para esto existe **`ColumnTransformer`**.

### 🧠 Analogía

Es como tener **dos máquinas en paralelo** en la fábrica:
- La máquina A solo procesa metal (numéricas)
- La máquina B solo procesa tela (categóricas)

El `ColumnTransformer` dirige cada columna a la máquina correcta y junta el resultado.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector

# Esta vez SI usamos la columna categórica 'origin'
mpg = sns.load_dataset('mpg').dropna()
X = mpg.drop(columns=['mpg', 'name'])
y = mpg['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Tipos de columnas:')
print(X.dtypes)

In [ ]:
# Forma 1: indicar columnas a mano
cols_num = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
cols_cat = ['origin']

preprocesador = make_column_transformer(
    (StandardScaler(),  cols_num),
    (OneHotEncoder(),   cols_cat),
    remainder='drop'   # qué hacer con columnas no listadas
)
preprocesador

In [ ]:
# Forma 2 (más elegante): usar selectores automáticos por tipo
preprocesador_auto = make_column_transformer(
    (StandardScaler(),  make_column_selector(dtype_include=np.number)),
    (OneHotEncoder(),   make_column_selector(dtype_include=object)),
    remainder='drop'
)
preprocesador_auto

# 4. El pipeline completo — ColumnTransformer + modelo

La magia: **un Pipeline puede contener un ColumnTransformer adentro**. Así juntamos TODO en un solo objeto.

In [ ]:
# Pipeline COMPLETO: preprocesamiento + modelo
pipeline_completo = make_pipeline(
    preprocesador_auto,
    LinearRegression()
)

pipeline_completo

In [ ]:
# UNA línea para entrenar TODO el flujo
pipeline_completo.fit(X_train, y_train)

# UNA línea para predecir
print(f'R² test: {pipeline_completo.score(X_test, y_test):.4f}')

# Predecir un caso nuevo — SIN preocuparte por escalar, codificar, nada
caso = X_test.iloc[[0]]
print(f'\nCaracterísticas del auto:')
print(caso)
print(f'\nPredicción de mpg: {pipeline_completo.predict(caso)[0]:.2f}')
print(f'Valor real:        {y_test.iloc[0]:.2f}')

# 5. Las grandes ventajas que acabas de ganar 🎁

Al usar Pipelines + ColumnTransformer:

| ✅ Ventaja | Explicación |
|---|---|
| **Cero data leakage** | `fit` aprende SOLO con train; `predict` aplica lo aprendido al test |
| **Reproducibilidad total** | Todo el flujo en un solo objeto que puedes guardar y cargar |
| **Código limpio** | De 6-8 líneas a 1 línea por fit/predict |
| **GridSearch fácil** | Puedes optimizar simultáneamente hiperparámetros del scaler Y del modelo |
| **Producción** | Lo que entrenas es exactamente lo que se despliega |

---
# 🏋️ Ejercicios prácticos

## Ejercicio 1 — Pipeline simple con `penguins`

**Tarea:** Predice el peso de un pingüino (`body_mass_g`) usando un **Pipeline**.

1. Cargar `sns.load_dataset('penguins').dropna()`.
2. Usar **solo numéricas** como features (excluir `species`, `island`, `sex`).
3. Train/test 80/20, `random_state=42`.
4. Crear un Pipeline con: `StandardScaler` + `LinearRegression`.
5. Entrenar y reportar R².
6. **Bonus:** comparar el R² con el de un modelo KNN (k=5) usando otro Pipeline.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline

pen = sns.load_dataset('penguins').dropna()
X = pen.select_dtypes(include='number').drop(columns=['body_mass_g'])
y = pen['body_mass_g']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe_lr  = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
pipe_knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)).fit(X_train, y_train)

print(f'Regresión Lineal → R² = {pipe_lr.score(X_test, y_test):.4f}')
print(f'KNN (k=5)        → R² = {pipe_knn.score(X_test, y_test):.4f}')
```
</details>

## Ejercicio 2 — ColumnTransformer con `tips`

Vamos a predecir la propina (`tip`) usando un dataset con columnas **mixtas**.

**Tarea:**

1. Cargar `sns.load_dataset('tips')`.
2. Target: `tip`.
3. Features: TODAS las demás (incluyendo `sex`, `smoker`, `day`, `time`).
4. Crear un **ColumnTransformer** que:
   - Aplique `StandardScaler` a las columnas numéricas.
   - Aplique `OneHotEncoder` a las categóricas.
5. Construir un Pipeline con: ColumnTransformer + `LinearRegression`.
6. Train/test split, entrenar y reportar R².

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector

tips = sns.load_dataset('tips')
X = tips.drop(columns=['tip'])
y = tips['tip']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocesador = make_column_transformer(
    (StandardScaler(),  make_column_selector(dtype_include=np.number)),
    (OneHotEncoder(),   make_column_selector(dtype_include='category'))  # tips usa category
)

pipe = make_pipeline(preprocesador, LinearRegression()).fit(X_train, y_train)
print(f'R² test: {pipe.score(X_test, y_test):.4f}')
```
</details>

## Ejercicio 3 — Pipeline completo + comparación de modelos

Vamos a comparar **3 modelos** usando el MISMO ColumnTransformer.

**Dataset:** `data/housing.csv` (lo tienen en su carpeta `data/`).

**Tarea:**

1. Cargar el dataset y aplicar `.dropna()`. Target: `median_house_value`.
2. Crear UN ColumnTransformer con:
   - `StandardScaler` para las columnas numéricas
   - `OneHotEncoder` para `ocean_proximity`
3. Crear **3 pipelines** usando el MISMO preprocesador:
   - Pipeline A: `LinearRegression`
   - Pipeline B: `KNeighborsRegressor(n_neighbors=10)`
   - Pipeline C: `DecisionTreeRegressor(max_depth=8, random_state=42)`
4. Train/test 80/20.
5. Entrenar los 3 y reportar R² + MAE + RMSE en una tabla.
6. **¿Cuál ganó?** Comenta por qué.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('data/housing.csv').dropna()
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ColumnTransformer reutilizable
preproc = make_column_transformer(
    (StandardScaler(),  make_column_selector(dtype_include=np.number)),
    (OneHotEncoder(),   make_column_selector(dtype_include=object))
)

# 3 pipelines compartiendo el preprocesador
modelos = {
    'Lineal': make_pipeline(preproc, LinearRegression()),
    'KNN(10)': make_pipeline(preproc, KNeighborsRegressor(n_neighbors=10)),
    'Árbol(d=8)': make_pipeline(preproc, DecisionTreeRegressor(max_depth=8, random_state=42))
}

resultados = []
for nombre, pipe in modelos.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    resultados.append({
        'Modelo': nombre,
        'R²':   r2_score(y_test, pred),
        'MAE':  mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred))
    })

print(pd.DataFrame(resultados).set_index('Modelo').round(2))
# 👉 KNN y Árbol suelen ganar al Lineal porque hay relaciones no-lineales
#    (la geografía afecta el precio de forma no-lineal).
```
</details>

---
## 📌 Cierre del día

Hoy aprendimos:

- ✅ Por qué los **Pipelines** son esenciales (no es opcional en producción)
- ✅ `make_pipeline` vs `Pipeline` con nombres
- ✅ **`ColumnTransformer`** para transformaciones distintas por tipo de columna
- ✅ Cómo combinar TODO en un solo objeto fit/predict
- ✅ El mismo Pipeline se puede reutilizar con distintos modelos

### 🔜 Mañana — Martes 26

- **SimpleImputer** para manejar valores nulos dentro del pipeline
- **Guardar y cargar modelos** con `joblib` para usarlos en producción

Nos vemos 🚀